In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("/Users/harishbabu/Documents/RAG/RAG_pipeline/.env"), override=True)

if os.getenv("OPENAPI_API_KEY"):
    print("API Key is set")
else:
    print("Not Found")

API Key is set


In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

In [38]:
response = llm.invoke("What is RAG?")
print(response.content)

RAG can mean different things depending on the context. The two most common meanings are:

- RAG status (Red-Amber-Green): A simple color-coded way to report project or risk status.
  - Green = on track or healthy
  - Amber (or yellow) = warning or potential issues
  - Red = serious problems or off track
  - Used in dashboards, project meetings, and risk reports to quickly convey status at a glance.

- Retrieval-Augmented Generation (RAG) in AI: A method that combines a knowledge retrieval step with a text generation model.
  - How it works: A retriever (e.g., a vector search system) fetches relevant documents from a large knowledge base, and a generator (e.g., a seq-to-seq model like BART/T5 or a GPT-style model) uses those documents to produce a grounded answer.
  - Benefits: Reduces hallucinations, provides grounded answers, handles up-to-date or domain-specific information.
  - Use cases: Open-domain question answering, customer support bots, knowledge-based assistants.

Which mean

## ** RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA

In [44]:
## Preparing Document for your text

from langchain_core.documents import Document

my_text = """
Accenture Chair and CEO Julie Sweet
“We delivered record second quarter bookings of $22.1 billion, including a record 41 clients with quarterly
bookings greater than $100 million, with revenues at the top of our guided range, while continuing to take
significant share in a competitive market. We’re accelerating our critical work with clients to scale advanced
AI across their enterprise, and we're seeing strong AI-driven growth. Our new strategic acquisitions will
further strengthen our capabilities and expand our scale to help clients create value and achieve AI-based
transformation. With our deep client relationships, industry and process expertise, leading and emerging
ecosystem partnerships, and unmatched execution strength, we are uniquely positioned to help clients
reinvent and capture the significant opportunities ahead.”
Second Quarter Fisca
"""
docs = [Document(page_content=my_text,metadata={"source":"Accenture","Context":"Accenture_CEO_statment"})]

In [45]:
## Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'Accenture', 'Context': 'Accenture_CEO_statment'}, page_content="Accenture Chair and CEO Julie Sweet\n“We delivered record second quarter bookings of $22.1 billion, including a record 41 clients with quarterly\nbookings greater than $100 million, with revenues at the top of our guided range, while continuing to take\nsignificant share in a competitive market. We’re accelerating our critical work with clients to scale advanced\nAI across their enterprise, and we're seeing strong AI-driven growth. Our new strategic acquisitions will"),
 Document(metadata={'source': 'Accenture', 'Context': 'Accenture_CEO_statment'}, page_content='further strengthen our capabilities and expand our scale to help clients create value and achieve AI-based\ntransformation. With our deep client relationships, industry and process expertise, leading and emerging\necosystem partnerships, and unmatched execution strength, we are uniquely positioned to help clients\nreinvent and captur

In [46]:
## Embeddings
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [47]:
embedding_model.embed_query("What is AI?")

[-0.0180206298828125,
 0.01250457763671875,
 -0.006984710693359375,
 0.00807952880859375,
 0.0162811279296875,
 -0.040618896484375,
 -0.009918212890625,
 -0.01311492919921875,
 -0.041351318359375,
 0.03033447265625,
 -0.0003383159637451172,
 -0.0472412109375,
 -0.02239990234375,
 -0.0556640625,
 -0.011749267578125,
 0.0036792755126953125,
 -0.0237884521484375,
 -0.0179290771484375,
 0.026885986328125,
 -0.0457763671875,
 0.004337310791015625,
 0.044921875,
 -0.01190948486328125,
 0.002513885498046875,
 0.0206146240234375,
 -0.03973388671875,
 0.035980224609375,
 0.004520416259765625,
 -0.026458740234375,
 -0.0142822265625,
 0.034210205078125,
 -0.0278778076171875,
 0.00450897216796875,
 -0.01654052734375,
 0.01319122314453125,
 0.025848388671875,
 -0.015625,
 0.01132965087890625,
 0.002803802490234375,
 -0.0215301513671875,
 0.01457977294921875,
 0.0057373046875,
 0.0228729248046875,
 0.043914794921875,
 -0.050994873046875,
 0.01007080078125,
 0.0007824897766113281,
 -0.021026611328125

In [56]:
## create and store embeddings in vector DataBase - ChromaDB

from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model
)

In [55]:
## This kinda transformation is done by langchain and stored it to chromaDB

vectors = {}

for doc in chunks:
    vector = embedding_model.embed_query(doc.page_content)
    vectors[tuple(vector)] = {
        "metadata": doc.metadata,
        "page_content": doc.page_content,
    }

first_vector, first_doc = next(iter(vectors.items()))
print(first_vector)
print(first_doc)

(0.035064697265625, -0.0293121337890625, 0.06396484375, -0.02069091796875, 0.02520751953125, -0.0157470703125, -0.0147857666015625, 0.07562255859375, -0.0328369140625, -0.011871337890625, 0.01384735107421875, -0.055389404296875, 0.0021991729736328125, -0.040924072265625, -0.018341064453125, 0.0253143310546875, -0.03057861328125, -0.03582763671875, 0.00926971435546875, -0.034698486328125, -0.0018606185913085938, 0.037322998046875, 0.03277587890625, 0.01178741455078125, 0.0080108642578125, 0.01262664794921875, -0.033447265625, -0.06268310546875, -0.025970458984375, -0.025970458984375, 0.02349853515625, -0.0201568603515625, 0.0179443359375, 0.0109710693359375, 0.0021762847900390625, -0.0025272369384765625, 0.00286102294921875, 0.040130615234375, -0.023162841796875, 0.01922607421875, 0.0111083984375, -0.0142974853515625, -0.00026154518127441406, 0.01593017578125, -0.0032482147216796875, 0.0209503173828125, 0.004268646240234375, -0.0304107666015625, -0.0211639404296875, 0.03973388671875, -0

In [60]:
#similarity_search
context = vectorstore.similarity_search("what did julie sweet say?",k=1)

In [68]:
responseWithoutRAG = llm.invoke("Who is Julie Sweet?")
print(responseWithoutRAG.content)

Julie Sweet is an American business executive who has served as the CEO of Accenture plc since 2019. Accenture is a global professional services company offering consulting, technology, and outsourcing services. Before becoming CEO, Sweet held various leadership roles within Accenture, including leading its North America business. If you’d like, I can add more details about her background or notable achievements.


In [69]:
responseWithRAG = llm.invoke(f"Who is Julie Sweet? You can answer using the following contex:{context}")
print(responseWithRAG.content)

Julie Sweet is the Chair and Chief Executive Officer (CEO) of Accenture, a global professional services company.


In [ ]:
reponse2WithoutRAG = llm.invoke("What is the second quarterly results of its fiscal year 2026 of Accenture in 2026?")
print(reponse2WithoutRAG.content)

I don’t have the 2026 Q2 results for Accenture in my built-in data. If you want the exact figures (revenue, net income, EPS, operating income, margins, etc.), I can pull them from the official release and summarize them for you, but I’d need the press release text or a link.

Tips to get it quickly:
- Go to Accenture’s Investor Relations site (investor.accenture.com).
- Look under News or Press Releases for “Second Quarter Fiscal 2026 Results.”
- The release will include the quarter’s revenue, net income, earnings per share (diluted), operating income, operating margin, and often guidance and notes on any one-time items.

If you paste the press release here or share the link, I’ll extract the key numbers and provide a concise summary and year-over-year changes.


In [74]:
response2WithRAG = llm.invoke(f"What is the second quarterly results of its fiscal year 2026 of Accenture? You can use the context: {context}")
print(response2.content)

The provided context does not include Accenture’s 2026 revenue figure. It mentions Q2 bookings of $22.1 billion and that revenues were at the top of their guided range, but no specific revenue amount or 2026 figure is given.

If you’d like, I can help locate the exact 2026 revenue from Accenture’s investor relations materials or a specific press release—just share the document or allow me to search for it.
